In [ ]:
# papermill parameters -- overridden at runtime by pipeline.py
ARTIFACT_DIR = ""
SAMPLE_DIR   = ""
MODEL_ID     = ""
SNAPSHOT_ID  = ""  # optional override; notebook resolves from manifest if empty


## Cél

A `lgbm_solusdt_s_fw60_2101_2605` short modell összes feature-jének prediktív erejét vizsgáljuk `ContinuousOptimalBinning` segítségével (max 10 bin per feature). Minden feature-re:
- Optimális bin-ekre osztjuk a feature értékeit (max 10 bin)
- A bin-enkénti target átlagot scatter ploton ábrázoljuk regressziós egyenessel
- IV (Information Value) és MI (Mutual Information) értéket számolunk a prediktív erő összehasonlíthatóságához

**Target:** `short_mfe_fw60` — short MFE 60 perces ablakban. Negatív értékek = profitábilis short mozgás.

**Struktúra:** Egy szekció = egy feature csoport. Minden csoportnál: IV tábla → korrelációs mátrix → tabset (egy fül = egy feature).
**Összefoglaló tábla** a plotok után.

In [ ]:
import sys
import io
import base64
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from IPython.display import display, Markdown
from optbinning import ContinuousOptimalBinning
from sklearn.feature_selection import mutual_info_regression

from analyst.lib.db_utils import find_repo_root
_root = find_repo_root()
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / "src"))

import utils
from analyst.lib.table_formatting import display_analysis_table, analysis_table_html
from analyst.lib.plot_utils import CQ_COLORS, setup_cq_theme
setup_cq_theme()

# ── Runtime paraméterek (papermill injektálja, fallback fejlesztéshez) ───────
MODEL_ID    = MODEL_ID    or "lgbm_solusdt_s_fw60_2101_2605"
SNAPSHOT_ID = SNAPSHOT_ID or ""   # üres — runtime-ban lesz feltöltve manifest-ből
ARTIFACT_DIR_PATH = Path(ARTIFACT_DIR) if ARTIFACT_DIR else (
    _root / "artifacts" / MODEL_ID
)
_direction  = MODEL_ID.split("_")[2] if len(MODEL_ID.split("_")) > 2 else "l"
TARGET      = "long_mfe_fw60" if _direction == "l" else "short_mfe_fw60"

# ── SNAPSHOT_ID fallback from manifest ───────────────────────────────────────
if not SNAPSHOT_ID:
    import json
    _manifest_path = ARTIFACT_DIR_PATH / "manifest.json"
    if _manifest_path.exists():
        _manifest = json.loads(_manifest_path.read_text(encoding="utf-8"))
        SNAPSHOT_ID = _manifest.get("snapshot_id", "")
    if not SNAPSHOT_ID:
        raise ValueError(f"SNAPSHOT_ID could not be resolved from manifest at {_manifest_path}")

# ── GROUP_MAP (same as long model — feature groups are model-agnostic) ────────
GROUP_MAP = {
    "feat_rsi_delta": "Accel", "feat_roc_delta": "Accel",
    "feat_return_momentum_delta": "Accel",
    "feat_rsi_x_trend": "Interaction", "feat_volume_x_momentum": "Interaction",
    "feat_rsi": "Momentum", "feat_roc": "Momentum", "feat_stoch_k": "Momentum",
    "feat_stoch_d": "Momentum", "feat_williams_r": "Momentum", "feat_cci": "Momentum",
    "feat_ema_slope_": "Trend Slope",
    "feat_sma_ratio": "Trend", "feat_ema_ratio": "Trend", "feat_wma_ratio": "Trend",
    "feat_kama_ratio": "Trend", "feat_macd_signal": "Trend", "feat_macd_diff": "Trend",
    "feat_macd": "Trend", "feat_adx_pos": "Trend", "feat_adx_neg": "Trend",
    "feat_adx": "Trend",
    "feat_bb_width": "Volatility", "feat_bb_position": "Volatility",
    "feat_atr_dist_high": "Market Structure", "feat_atr_dist_low": "Market Structure",
    "feat_atr": "Volatility", "feat_natr": "Volatility", "feat_hist_vol": "Volatility",
    "feat_parkinson_vol": "Volatility", "feat_gk_vol": "Volatility",
    "feat_volume_confirmed_return_": "Volume",
    "feat_volume_accel_": "Volume", "feat_volume_rank_": "Volume",
    "feat_volume_sma": "Volume", "feat_volume_ratio": "Volume",
    "feat_obv_roc": "Volume", "feat_obv": "Volume",
    "feat_mfi": "Volume", "feat_ad_line": "Volume", "feat_cmf": "Volume",
    "feat_returns_log": "Price Action", "feat_returns_sma": "Price Action",
    "feat_returns_std": "Price Action", "feat_returns_skew": "Price Action",
    "feat_returns_kurt": "Price Action",
    "feat_hml_range": "Price Action", "feat_ohlc_range": "Price Action",
    "feat_close_position": "Price Action",
    "feat_prev_session_high_dist": "Market Structure",
    "feat_prev_session_low_dist": "Market Structure",
    "feat_rolling_drawdown_": "Market Structure",
    "feat_dist_rolling_": "Market Structure",
    "feat_range_expansion_": "Market Structure",
    "feat_recovery_ratio": "Market Structure", "feat_max_drawdown": "Market Structure",
    "feat_time_since_high": "Market Structure", "feat_time_since_low": "Market Structure",
    "feat_efficiency_ratio": "Market Structure",
    "feat_regime_rank": "Regime Rank", "feat_return_autocorr": "Autocorrelation",
    "feat_variance_ratio": "Autocorrelation",
    "feat_lower_wick_ratio": "Candle Pattern", "feat_upper_wick_ratio": "Candle Pattern",
    "feat_signed_body_": "Candle Pattern", "feat_wick_imbalance_": "Candle Pattern",
    "feat_doji": "Candle Pattern", "feat_hammer": "Candle Pattern",
    "feat_shooting_star": "Candle Pattern", "feat_inside_bar": "Candle Pattern",
    "feat_outside_bar": "Candle Pattern", "feat_engulf_bull": "Candle Pattern",
    "feat_engulf_bear": "Candle Pattern", "feat_bull_bars_ratio": "Candle Pattern",
    "feat_body_ratio": "Candle Pattern", "feat_wick_ratio": "Candle Pattern",
    "feat_lr_slope": "Trend Slope", "feat_lr_r2": "Trend Slope",
    "feat_lr_residual": "Trend Slope", "feat_trend_slope": "Trend Slope",
    "feat_pos_return_mean": "Tail Risk", "feat_neg_return_mean": "Tail Risk",
    "feat_return_asymmetry": "Tail Risk",
    "feat_gap_open_abs_sma": "Gap", "feat_gap_open": "Gap",
    "feat_tenkan_kijun_": "Ichimoku", "feat_price_vs_": "Ichimoku",
    "feat_tenkan_ratio": "Ichimoku", "feat_kijun_ratio": "Ichimoku",
    "feat_senkou_b_ratio": "Ichimoku", "feat_ichimoku_cloud_thickness": "Ichimoku",
    "feat_donchian_width": "Donchian", "feat_donchian_position": "Donchian",
    "feat_donchian_breakout": "Donchian",
    "feat_hour_sin": "Time/Session", "feat_hour_cos": "Time/Session",
    "feat_dayofweek_sin": "Time/Session", "feat_dayofweek_cos": "Time/Session",
    "feat_weekend": "Time/Session", "feat_session_asia": "Time/Session",
    "feat_session_europe": "Time/Session", "feat_session_us": "Time/Session",
    "feat_bars_into_session_norm": "Time/Session", "feat_day_range_position": "Time/Session",
    "feat_day_open_return": "Time/Session", "feat_weekly_open_return": "Time/Session",
    "feat_dist_sma": "Return Distance", "feat_dist_bb": "Return Distance",
    "feat_return_z_": "Price Action", "feat_vol_adj_return_": "Price Action",
    "feat_return_": "Price Action",
    "feat_higher_high_": "Swing", "feat_higher_low_": "Swing",
    "feat_lower_high_": "Swing", "feat_lower_low_": "Swing",
    "feat_swing_": "Swing", "feat_directional_agreement_": "Swing",
    "feat_taker_": "Taker Flow", "feat_avg_trade_": "Taker Flow",
    "feat_trade_count_": "Taker Flow", "feat_quote_volume_": "Taker Flow",
}

def assign_group(feat_name):
    for prefix, group in GROUP_MAP.items():
        if feat_name.startswith(prefix):
            return group
    return "Other"

# ── Data load ─────────────────────────────────────────────────────────────────
conn = utils.open_lab_connection("solusdt")
df_all = conn.execute(f"""
    SELECT s.*, m.split
    FROM snap."{SNAPSHOT_ID}" AS s
    INNER JOIN model."{MODEL_ID}__sample" AS m
        ON s.open_time = m.open_time
    ORDER BY s.open_time
""").fetchdf()
conn.close()

feat_cols = [c for c in df_all.columns if c.startswith("feat_")]
df_train  = df_all[df_all["split"] == 0].copy()

groups_df = pd.DataFrame([
    {"Group": assign_group(f), "Feature": f} for f in sorted(feat_cols)
]).sort_values(["Group", "Feature"]).reset_index(drop=True)

_null_rates = df_train[feat_cols].isnull().mean()
_inf_rates  = df_train[feat_cols].apply(lambda c: c.apply(lambda v: v == float('inf') or v == float('-inf')).mean())
_variances  = df_train[feat_cols].var()
_quality_bad = set(
    _null_rates[_null_rates > 0.01].index.tolist() +
    _inf_rates[_inf_rates > 0.001].index.tolist() +
    _variances[_variances < 1e-8].index.tolist()
)
feat_cols_quality_dropped = sorted(_quality_bad)
feat_cols = [f for f in feat_cols if f not in _quality_bad]
display(Markdown(f"**Quality szűrő:** {len(feat_cols_quality_dropped)} feature kizárva | Maradék: {len(feat_cols)}"))
n_other = (groups_df["Group"] == "Other").sum()
display(Markdown(
    f"**Adatbetöltés kész** | Train sorok: {len(df_train):,} "
    f"| Feature-ök: {len(feat_cols)} | Csoportok: {groups_df['Group'].nunique()} | Other: {n_other}"
))


In [ ]:
#| label: tbl-iv-precompute
#| echo: false

iv_results = {}

for feat in feat_cols:
    x = df_train[feat].dropna()
    y = df_train.loc[x.index, TARGET]
    if len(x) < 50:
        iv_results[feat] = {"iv": float("nan"), "n_bins": 0, "bin_data": None}
        continue
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ob = ContinuousOptimalBinning(name=feat, max_n_bins=10)
            ob.fit(x.values, y.values)
            bt = ob.binning_table.build()
            bin_data = bt[~bt["Bin"].isin(["Special", "Missing"])].copy()
            iv_results[feat] = {"iv": ob.binning_table.iv, "n_bins": len(bin_data),
                                "bin_data": bin_data, "ob": ob}
    except Exception:
        iv_results[feat] = {"iv": float("nan"), "n_bins": 0, "bin_data": None}

n_ok = sum(1 for v in iv_results.values() if v["bin_data"] is not None)
print(f"IV számítás kész: {n_ok}/{len(feat_cols)} feature OK")

_X_mi = df_train[feat_cols].fillna(df_train[feat_cols].median())
_valid_mi = df_train[TARGET].notna()
_mi_vals = mutual_info_regression(
    _X_mi[_valid_mi], df_train.loc[_valid_mi, TARGET],
    discrete_features=False, random_state=42,
)
_mi_dict = dict(zip(feat_cols, _mi_vals))
for _f in feat_cols:
    iv_results[_f]["mi"] = round(_mi_dict.get(_f, float("nan")), 5)

n_ok_mi = sum(1 for v in iv_results.values() if not (v.get("mi") != v.get("mi")))
print(f"MI számítás kész: {n_ok_mi}/{len(feat_cols)} feature OK")


In [ ]:
#| output: asis
#| echo: false
#| warning: false

TABSET_OPEN  = "::: {.panel-tabset}"
TABSET_CLOSE = ":::"

GROUP_ORDER = [
    "Price Action", "Volatility", "Volume", "Gap", "Candle Pattern",
    "Time/Session", "Taker Flow", "Momentum", "Trend", "Autocorrelation",
    "Market Structure", "Swing", "Tail Risk", "Ichimoku", "Donchian",
    "Trend Slope", "Accel",
]
_available = set(groups_df["Group"].unique())
_ordered_groups = [g for g in GROUP_ORDER if g in _available] + \
                  sorted(g for g in _available if g not in set(GROUP_ORDER))

MI_THRESHOLD   = 0.001
CORR_THRESHOLD = 0.98

_fs_selected = []
_fs_dropped_mi   = {}
_fs_dropped_corr = {}
_fs_dropped_qual = {f: 'quality: null/inf/variance' for f in feat_cols_quality_dropped}

for group in _ordered_groups:
    feats_in_group = sorted(groups_df[groups_df["Group"] == group]["Feature"].tolist())
    feats_mi_ok = [f for f in feats_in_group
                   if iv_results.get(f, {}).get("mi", 0) > MI_THRESHOLD]
    corr_dedup = {}
    if len(feats_mi_ok) >= 2:
        _cm = df_train[feats_mi_ok].corr().abs()
        _seen = set()
        for i, fa in enumerate(feats_mi_ok):
            if fa in _seen:
                continue
            for fb in feats_mi_ok[i+1:]:
                if fb in _seen:
                    continue
                if _cm.loc[fa, fb] >= CORR_THRESHOLD:
                    mi_a = iv_results.get(fa, {}).get("mi", 0)
                    mi_b = iv_results.get(fb, {}).get("mi", 0)
                    drop = fb if mi_a >= mi_b else fa
                    corr_dedup[drop] = fa if mi_a >= mi_b else fb
                    _seen.add(drop)
    feats_pass = [f for f in feats_mi_ok if f not in corr_dedup]
    for _f in feats_in_group:
        _mi_val = iv_results.get(_f, {}).get('mi', 0)
        if _mi_val != _mi_val or _mi_val <= MI_THRESHOLD:
            _fs_dropped_mi[_f] = f'mi <= {MI_THRESHOLD} (mi={_mi_val:.5f})'
        elif _f in corr_dedup:
            _fs_dropped_corr[_f] = f'duplikatum -> {corr_dedup[_f]}'
        else:
            _fs_selected.append(_f)

    def _dontes(f):
        mi_val = iv_results.get(f, {}).get("mi", float("nan"))
        if mi_val != mi_val or mi_val <= MI_THRESHOLD:
            return "✗ kizár (MI)"
        if f in corr_dedup:
            return f"✗ duplikátum → {corr_dedup[f].replace('feat_','')}"
        return ""

    print(f"\n## {group}\n")
    group_iv_rows = []
    for f in feats_in_group:
        res = iv_results.get(f, {})
        mi_val = res.get("mi", float("nan"))
        group_iv_rows.append({"Feature": f, "IV": round(res.get("iv", float("nan")), 5),
                               "MI": round(mi_val, 5), "N_bins": res.get("n_bins", 0), "Döntés": _dontes(f)})
    group_iv_df = (pd.DataFrame(group_iv_rows).sort_values("MI", ascending=False).reset_index(drop=True))
    print(analysis_table_html(group_iv_df))

    if len(feats_pass) >= 2:
        corr_data = df_train[feats_pass].corr()
        n = len(feats_pass)
        cell_sz = 0.55; margin = 2.0
        fig_h = max(4.0, n * cell_sz + margin)
        fig_w = max(5.0, n * cell_sz + margin + 1.0)
        lbl_sz = max(6, min(9, 90 // n)); ann_sz = max(6, min(9, 100 // n))
        mask = np.triu(np.ones_like(corr_data, dtype=bool), k=1)
        with plt.rc_context({"axes.grid": False}):
            fig_c, ax_c = plt.subplots(figsize=(fig_w, fig_h))
            sns.heatmap(corr_data, ax=ax_c, mask=mask, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                        annot=(n <= 20), fmt=".2f", annot_kws={"size": ann_sz}, linewidths=0, square=True,
                        cbar_kws={"shrink": 0.6},
                        xticklabels=[f.replace("feat_", "") for f in feats_pass],
                        yticklabels=[f.replace("feat_", "") for f in feats_pass])
            ax_c.tick_params(axis="x", rotation=45, labelsize=lbl_sz)
            ax_c.tick_params(axis="y", rotation=0, labelsize=lbl_sz)
            fig_c.tight_layout()
        buf_c = io.BytesIO()
        fig_c.savefig(buf_c, format="png", dpi=110, bbox_inches="tight")
        plt.close(fig_c); buf_c.seek(0)
        img_c_b64 = base64.b64encode(buf_c.read()).decode("utf-8")
        print(f'<img src="data:image/png;base64,{img_c_b64}" style="max-width:900px; margin-bottom:1rem;"/>\n')

    if not feats_pass:
        print("*Minden feature kizárva — nincs megjeleníthető plot.*\n")
    else:
        print(TABSET_OPEN + "\n")
        for feat in feats_pass:
            print(f"\n### {feat}\n")
            res = iv_results.get(feat, {}); ob = res.get("ob")
            if ob is None:
                print("*Nem elégséges adat a binninghez.*\n"); continue
            _show_bak = plt.show; plt.show = lambda *a, **kw: None
            try:
                ob.binning_table.plot(add_special=False, add_missing=False, show_bin_labels=True, figsize=(9, 4))
                fig = plt.gcf()
                for ax in fig.get_axes(): ax.grid(False)
                fig.tight_layout()
            finally:
                plt.show = _show_bak
            buf = io.BytesIO()
            fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
            plt.close(fig); buf.seek(0)
            img_b64 = base64.b64encode(buf.read()).decode("utf-8")
            print(f'<img src="data:image/png;base64,{img_b64}" style="max-width:820px;"/>\n')
        print(TABSET_CLOSE + "\n")


## Összefoglaló — Feature szelekció eredménye

Csoport-bajnokok (legjobb MI), Top 10, és teljes kizárási lista MI szerint rendezve.


In [ ]:
#| label: tbl-summary
#| echo: false

import pandas as _pd

_summary_rows = []
for _f in feat_cols:
    _mi = iv_results.get(_f, {}).get("mi", float("nan"))
    _iv = iv_results.get(_f, {}).get("iv", float("nan"))
    _grp = assign_group(_f)
    if _f in _fs_dropped_corr:
        _dec = f"duplikátum → {_fs_dropped_corr[_f].replace('feat_', '')}"
    elif _f in _fs_dropped_mi:
        _dec = f"MI ≤ {MI_THRESHOLD}"
    else:
        _dec = "✓ selected"
    _summary_rows.append({"feature": _f.replace("feat_", ""), "group": _grp,
                           "mi": _mi, "iv": _iv, "decision": _dec, "full_name": _f})
for _f in feat_cols_quality_dropped:
    _summary_rows.append({"feature": _f.replace("feat_", ""), "group": assign_group(_f),
                           "mi": float("nan"), "iv": float("nan"),
                           "decision": "quality: null/inf/variance", "full_name": _f})

summary_df = _pd.DataFrame(_summary_rows).sort_values("mi", ascending=False, na_position="last")
summary_df["mi_disp"] = summary_df["mi"].apply(lambda x: f"{x:.4f}" if x == x else "—")
summary_df["iv_disp"] = summary_df["iv"].apply(lambda x: f"{x:.5f}" if x == x else "—")
_sel_df = summary_df[summary_df["decision"] == "✓ selected"].copy()
top10_feats = _sel_df["full_name"].head(10).tolist()
_grp_best_name = {}; _grp_best_feat = {}
for _, _row in _sel_df.iterrows():
    _g = _row["group"]
    if _g not in _grp_best_feat:
        _grp_best_feat[_g] = _row["full_name"]
        _grp_best_name[_g] = _row["feature"]
top1_per_group_feats = list(_grp_best_feat.values())

_champ_rows = [f"| {_g} | {_fname} | {_sel_df[_sel_df['feature'] == _fname]['mi_disp'].values[0]} |"
               for _g, _fname in _grp_best_name.items()]
display(Markdown(
    f"### Csoport bajnokok ({len(_grp_best_feat)} csoport)\n\n"
    "| Csoport | Feature | MI |\n|---------|---------|-----|\n" + "\n".join(_champ_rows)
))
_top10_rows = [f"| {i+1} | {r['feature']} | {r['group']} | {r['mi_disp']} | {r['iv_disp']} |"
               for i, (_, r) in enumerate(_sel_df.head(10).iterrows())]
display(Markdown(
    "### Top 10 feature (MI szerint)\n\n"
    "| # | Feature | Csoport | MI | IV |\n|---|---------|---------|-----|-----|\n" + "\n".join(_top10_rows)
))
_all_rows = [f"| {r['feature']} | {r['group']} | {r['mi_disp']} | {r['iv_disp']} | {r['decision']} |"
             for _, r in summary_df.iterrows()]
display(Markdown(
    f"### Összes feature MI szerint rendezve ({len(summary_df)} db)\n\n"
    "| Feature | Csoport | MI | IV | Döntés |\n|---------|---------|-----|-----|--------|\n" + "\n".join(_all_rows)
))


In [ ]:
#| label: tbl-output-summary
#| echo: false

import json as _json
from datetime import datetime as _dt

_created_at = _dt.now().strftime('%Y-%m-%d %H:%M:%S')
_run_id     = _dt.now().strftime('run_%Y%m%d_%H%M%S')
_all_dropped = []
for _f, _r in _fs_dropped_qual.items(): _all_dropped.append({'col': _f, 'reason': _r})
for _f, _r in _fs_dropped_mi.items():
    if _f not in _fs_dropped_qual: _all_dropped.append({'col': _f, 'reason': _r})
for _f, _r in _fs_dropped_corr.items(): _all_dropped.append({'col': _f, 'reason': _r})
_asset_id   = MODEL_ID.split('_')[1] if len(MODEL_ID.split('_')) > 1 else 'solusdt'
_direction  = MODEL_ID.split('_')[2] if len(MODEL_ID.split('_')) > 2 else 'l'
_target_col = 'long_mfe_fw60' if _direction == 'l' else 'short_mfe_fw60'
feature_set = {
    'run_id': _run_id, 'asset_id': _asset_id, 'model_id': MODEL_ID, 'target_col': _target_col,
    'created_at': _created_at, 'target_cols': [_target_col],
    'selected': sorted(_fs_selected), 'top10': top10_feats, 'top1_per_group': top1_per_group_feats,
    'dropped': _all_dropped, 'review': [],
    'provenance': {'snapshot_id': SNAPSHOT_ID, 'sample_rows': len(df_train),
                   'min_open_time': str(df_train['open_time'].min()),
                   'max_open_time': str(df_train['open_time'].max())},
    'thresholds': {'mi_threshold': MI_THRESHOLD, 'corr_threshold': CORR_THRESHOLD,
                   'max_null_rate': 0.01, 'max_inf_rate': 0.001, 'min_variance': 1e-8},
}
_fe_dir = ARTIFACT_DIR_PATH / 'feature_engineering'
_fe_dir.mkdir(parents=True, exist_ok=True)
_out_path = _fe_dir / 'feature_set.json'
_out_path.write_text(_json.dumps(feature_set, indent=4, ensure_ascii=False), encoding='utf-8')
display(Markdown(
    f'**Feature szelekció eredménye**\n\n'
    f'| Kategória | Darab |\n|-----------|-------|\n'
    f'| selected | {len(feature_set["selected"])} |\n'
    f'| top10 | {len(feature_set["top10"])} |\n'
    f'| top1_per_group | {len(feature_set["top1_per_group"])} |\n'
    f'| dropped (quality) | {len(_fs_dropped_qual)} |\n'
    f'| dropped (MI) | {len(_fs_dropped_mi)} |\n'
    f'| dropped (duplikátum) | {len(_fs_dropped_corr)} |\n'
    f'\n**Output:** `{_out_path}` \u2713'
))
